# 01 — Exploratory Data Analysis

This notebook:
1. Loads the dataset from Hugging Face and filters it to 40 classes
2. Shows a 5×8 grid of sample images (one per class)
3. Plots the class distribution
4. Prints the train/val/test split sizes

The first run downloads dataset(~5 GB) into `data/` — subsequent runs use the cache.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from config import Config
from utils import set_seed
from data import FOOD40, load_food40_splits

set_seed(Config.SEED)

print(f"Device: {Config.DEVICE}")
print(f"Number of classes: {len(FOOD40)}")

In [ ]:
train_ds, val_ds, test_ds, label_map = load_food40_splits()

inv_label_map = {v: k for k, v in label_map.items()}

print("Split created successfully.")

In [ ]:
n_train, n_val, n_test = len(train_ds), len(val_ds), len(test_ds)
n_total = n_train + n_val + n_test

print(f"Train: {n_train:>6} images  ({n_train / n_total:.1%})")
print(f"Val:   {n_val:>6} images  ({n_val / n_total:.1%})")
print(f"Test:  {n_test:>6} images  ({n_test / n_total:.1%})")
print(f"Total: {n_total:>6} images across {len(FOOD40)} classes")

In [ ]:
labels_col = train_ds["label"]
first_idx = {}
for pos, orig_label in enumerate(labels_col):
    cid = label_map[orig_label]
    if cid not in first_idx:
        first_idx[cid] = pos
    if len(first_idx) == len(FOOD40):
        break 

fig, axes = plt.subplots(5, 8, figsize=(20, 13))
for cid, ax in enumerate(axes.flat):
    item = train_ds[first_idx[cid]]
    ax.imshow(item["image"].convert("RGB"))
    ax.set_title(FOOD40[cid].replace("_", " "), fontsize=9)
    ax.axis("off")

fig.suptitle("One sample per class — Food-40 training set", fontsize=16)
fig.tight_layout()
plt.show()

In [ ]:
counts = pd.Series(labels_col).map(label_map).value_counts().sort_index()
counts.index = [FOOD40[i].replace("_", " ") for i in counts.index]

plt.figure(figsize=(16, 6))
sns.barplot(x=counts.index, y=counts.values, color="#4c72b0")
plt.xticks(rotation=75, ha="right", fontsize=9)
plt.ylabel("Training images")
plt.title("Class distribution — Food-40 training split")
plt.tight_layout()
plt.show()

print(f"Min class count: {counts.min()}  |  Max class count: {counts.max()}")